# Kategorizácia textu podľa `general_criterion`
V tomto notebooku natrénujeme jednoduchý model, ktorý na základe stĺpca `description` predikuje `general_criterion`.

In [ ]:
# Importy
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import ipywidgets as widgets
from IPython.display import display, clear_output

# 3. Načítanie a príprava dát
data_path = 'data/contract_criteria_final_general_only.csv'
df = pd.read_csv(data_path)
df = df.dropna(subset=['description', 'general_criterion'])
df['description'] = df['description'].astype(str).str.strip()

# Mapovanie priamych záznamov pre presné priradenie
desc_to_label = {desc.lower(): lab for desc, lab in zip(df['description'], df['general_criterion'])}

print("Prvých pár riadkov datasetu:")
print(df.head(3), "\n")

# 4. Rozdelenie na tréning a test
X_train, X_test, y_train, y_test = train_test_split(
    df['description'], df['general_criterion'], test_size=0.2, random_state=42
)
print(f'Tréning: {len(X_train)} vzoriek, Test: {len(X_test)} vzoriek\n')

# 5. Tréning modelov
pipeline_nb = Pipeline([('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])
pipeline_lr = Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])

pipeline_nb.fit(X_train, y_train)
pipeline_lr.fit(X_train, y_train)

# 6. Vyhodnotenie
def report(name, model):
    preds = model.predict(X_test)
    print(f"=== Report pre {name} ===")
    print(classification_report(y_test, preds))

print("Vyhodnotenie modelov:")
report('Naive Bayes', pipeline_nb)
report('LogisticRegression', pipeline_lr)

# 7. Interaktívna predikcia s presným záznamom + fallback
model_selector = widgets.ToggleButtons(options=[('Naive Bayes', 'nb'), ('LogisticReg', 'lr')],
                                        description='Model:')
text_input = widgets.Text(placeholder='Zadaj popis... (presne alebo nový)', description='Popis:',
                          layout=widgets.Layout(width='70%'))
predict_button = widgets.Button(description='Predikovať', button_style='info')
output = widgets.Output()


def on_predict_clicked(b):
    with output:
        clear_output()
        text = text_input.value.strip()
        if not text:
            print('Prosím, zadaj popis.')
            return
        key = text.lower()
        # Priame presné priradenie z datasetu
        if key in desc_to_label:
            print(f"Presné priradenie z datasetu: {desc_to_label[key]}")
        else:
            # fallback na model
            if model_selector.value == 'nb':
                pred = pipeline_nb.predict([text])[0]
            else:
                pred = pipeline_lr.predict([text])[0]
            print(f"Predikovaná kategória ({model_selector.value}): {pred}")

predict_button.on_click(on_predict_clicked)

display(widgets.VBox([model_selector, widgets.HBox([text_input, predict_button]), output]))


Prvých pár riadkov datasetu:
   contract_id                                          criterion  \
0      3254340                             Lehota realizácie prác   
1      3254631  Počet kalendárnych dní realizácie stavebnych prác   
2      3254635                            Lehota výstavby v dňoch   

                                         description  \
0  Projekt rieši stavebné úpravy pre zníženie ene...   
1  Miestne kultúrne stredisko je samostatne stoja...   
2  Predmetom zákazky je komplexná rekonštrukcia b...   

                            label  general_criterion  
0                  Stavebné práce  Čas / harmonogram  
1                  Stavebné práce  Čas / harmonogram  
2  Stavebné práce na stavbe budov  Čas / harmonogram   

Tréning: 1716 vzoriek, Test: 430 vzoriek

Vyhodnotenie modelov:
=== Report pre Naive Bayes ===
                                     precision    recall  f1-score   support

              Bezpečnostné hľadisko       0.00      0.00      0.00        

c:\Users\marek\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\marek\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\marek\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif